In [1]:
%load_ext autoreload
%autoreload 2

In [23]:
import pandas as pd
import numpy as np

In [ ]:
import orekit_jpype as orekit


orekit.initVM(jvmpath="C://Program Files//Eclipse Adoptium//jdk-17.0.18.8-hotspot//bin//server//jvm.dll",
              additional_classpaths=[r"D:\GitHub\contigo_edr\java_src\target\orekit_utils-1.0.0.jar"])


In [3]:
from orekit_jpype.pyhelpers import setup_orekit_data, download_orekit_data_curdir
#download_orekit_data_curdir()
setup_orekit_data(from_pip_library=False)

In [33]:
from contigo.constellation import Constellation
from contigo.ephemeris.orekit_ephem import OrekitEphem
from contigo.ephemeris.spice_ephem import SPICEEphem
from contigo.solar_system_ephem import SolarSystemEnvironment
from contigo.forces.third_body_acc import ThirdBodyEnv


In [5]:
# read in data
sw_d = pd.read_hdf('./data/ESA_pod.hdf')
ore_d = pd.read_hdf('./data/ore_d.hdf')
ore_a = ore_d[[ 'ecef_sn_ax', 'ecef_sn_ay', 'ecef_sn_az', 
               'ecef_mn_ax', 'ecef_mn_ay', 'ecef_mn_az']].to_numpy()
hdf_sc = Constellation(state_file=r'./data/ESA_pod.hdf',
                    time_col='DateTime', x_col='x', y_col='y', z_col='z',
                    vx_col='vx', vy_col='vy', vz_col='vz',
                    sc_id_col='filename', sc_fn_slc=slice(-11,-8),
                    tscale_input='GPS',
                    sc_mass=4.3e+02, cr=1.8, srp_area=1)


In [16]:
s_ephem = SPICEEphem()
s_env = SolarSystemEnvironment(['SUN','MOON'], 
                             ephem_time=hdf_sc.sspice_et[0:500], 
                             gps_time=hdf_sc.sspice_gps[0:500],
                             utc_time=hdf_sc.sc_utc[0:500],
                             tolerance=0.001,provider=s_ephem)

In [27]:
o_ephem = OrekitEphem()
o_env = SolarSystemEnvironment(['SUN','MOON'], 
                             ephem_time=hdf_sc.sspice_et[0:500], 
                             gps_time=hdf_sc.sspice_gps[0:500],
                             utc_time=hdf_sc.sc_utc[0:500],
                             tolerance=1,provider=o_ephem)

In [ ]:
_, _, _, t1 = o_env.get_ephem(ephem_time=hdf_sc.sspice_et[0:500], 
                             gps_time=hdf_sc.sspice_gps[0:500],
                             utc_time=hdf_sc.sc_utc[0:500])
_, _, _, t2 = s_env.get_ephem(ephem_time=hdf_sc.sspice_et[0:500], 
                             gps_time=hdf_sc.sspice_gps[0:500],
                             utc_time=hdf_sc.sc_utc[0:500])

print(t1.shape)
print(t2.shape)

In [31]:
_, _, _, t1 = o_env.get_ephem(ephem_time=hdf_sc.sspice_et, 
                             gps_time=hdf_sc.sspice_gps,
                             utc_time=hdf_sc.sc_utc)
_, _, _, t2 = s_env.get_ephem(ephem_time=hdf_sc.sspice_et, 
                             gps_time=hdf_sc.sspice_gps,
                             utc_time=hdf_sc.sc_utc)

print(t1.shape)
print(t2.shape)

(2, 138240, 3)
(2, 138240, 3)


In [37]:
ore_acc = ThirdBodyEnv( ).acceleration(hdf_sc, o_env)
spi_acc = ThirdBodyEnv( ).acceleration(hdf_sc, s_env)

In [43]:
print(ore_acc['ESA'].shape)
print(spi_acc['ESA'].shape)

print(np.allclose(ore_acc['ESA'], spi_acc['ESA'], atol=1e-7))
print(ore_acc['ESA'][0,0,:])
print(spi_acc['ESA'][0,0,:])

(2, 138240, 3)
(2, 138240, 3)
True
[-9.00593698e-11  2.80000067e-10  9.67685633e-12]
[-9.00593868e-11  2.80000065e-10  9.67684671e-12]


In [ ]:
from org.orekit.frames import FramesFactory
from org.orekit.time import TimeScalesFactory, AbsoluteDate
from org.orekit.utils import IERSConventions, Constants, PVCoordinates
from org.orekit.orbits import CartesianOrbit
from org.orekit.propagation import SpacecraftState
from org.orekit.forces.radiation import SolarRadiationPressure, IsotropicRadiationSingleCoefficient
from org.orekit.bodies import CelestialBodyFactory, OneAxisEllipsoid
from org.hipparchus.geometry.euclidean.threed import Vector3D

utc = TimeScalesFactory.getUTC()
dt = hdf_sc.sc_utc
sc_pos = hdf_sc['ESA'].state*1000. # convert to meters for orekit

epoch = [AbsoluteDate(dt.year, dt.month, dt.day, dt.hour, dt.minute, float(dt.second), utc)
          for dt in sw_d['DateTime']]

srp_ecef = np.zeros((sc_pos.shape[0], 3))
srp_eci = np.zeros((sc_pos.shape[0], 3))

for i in np.arange(0, sc_pos.shape[0]):

    # 1. Setup Frames
    ecef = FramesFactory.getITRF(IERSConventions.IERS_2010, True)
    eci = FramesFactory.getEME2000()
    date = epoch[i]

    # 2. Your ECEF State (Example values in meters and m/s)
    position_ecef = Vector3D(sc_pos[i,0], sc_pos[i,1], sc_pos[i,2])
    velocity_ecef = Vector3D(sc_pos[i,3], sc_pos[i,4], sc_pos[i,5])
    pv_ecef = PVCoordinates(position_ecef, velocity_ecef)

    # 3. Create State in GCRF (Orekit's physics engine expects inertial)
    # We transform the ECEF PV to GCRF first
    transform = ecef.getTransformTo(eci, date)
    pv_eci = transform.transformPVCoordinates(pv_ecef)
    orbit_eci = CartesianOrbit(pv_eci, eci, date, Constants.WGS84_EARTH_MU)
    state = SpacecraftState(orbit_eci).withMass(430)

    # 4. Define SRP Model
    sun = CelestialBodyFactory.getSun()
    earth_radius = OneAxisEllipsoid(Constants.WGS84_EARTH_EQUATORIAL_RADIUS,
                                Constants.WGS84_EARTH_FLATTENING,
                                ecef)

    # Isotropic model: Area = 10.0m^2, Cr = 1.2
    satellite_surface = IsotropicRadiationSingleCoefficient(1.0, 1.8)
    srp_model = SolarRadiationPressure(sun, earth_radius, satellite_surface)

    # 5. Calculate Acceleration in Inertial Frame
    accel_eci = srp_model.acceleration(state, srp_model.getParameters())

    # 6. Rotate Acceleration back to ECEF
    # Note: Since acceleration is a vector, we only need the rotation component
    rotation_eci_to_ecef = eci.getStaticTransformTo(ecef, date).getRotation()
    accel_ecef = rotation_eci_to_ecef.applyTo(accel_eci)

    srp_ecef[i,0] = accel_ecef.getX()
    srp_ecef[i,1] = accel_ecef.getY()
    srp_ecef[i,2] = accel_ecef.getZ()

    srp_eci[i,0] = accel_eci.getX()
    srp_eci[i,1] = accel_eci.getY()
    srp_eci[i,2] = accel_eci.getZ()

NameError: name 'utc' is not defined

In [82]:
sw_d['DateTime'][0]

Timestamp('2018-01-02 00:00:00')

In [85]:
print(sw_d[['srp_x','srp_y','srp_z']].to_numpy()[0]*1000.)
print(np.array([accel_eci.getX(), accel_eci.getY(), accel_eci.getZ()]))

[-3.86205902e-09  1.77634948e-08  7.70034738e-09]
[-3.86228622e-09  1.77641755e-08  7.70064182e-09]


In [ ]:
accel_eci.get

array([[-3.08859041e+02, -6.75958258e+03, -1.27683430e+03,
        -1.25730908e-01,  1.40789886e+00, -7.47203275e+00],
       [-3.10087065e+02, -6.74509173e+03, -1.35147486e+03,
        -1.19851459e-01,  1.49024109e+00, -7.45592753e+00],
       [-3.11255623e+02, -6.72977841e+03, -1.42594981e+03,
        -1.13837752e-01,  1.57238768e+00, -7.43890976e+00],
       ...,
       [ 2.60666729e+02, -7.84852317e+01, -6.88361428e+03,
         3.01717561e+00,  6.94813616e+00,  3.58674454e-02],
       [ 2.90872810e+02, -9.02254620e+00, -6.88283716e+03,
         3.02397882e+00,  6.94425955e+00,  1.19554905e-01],
       [ 3.21145068e+02,  6.03971333e+01, -6.88122323e+03,
         3.03041072e+00,  6.93953513e+00,  2.03228431e-01]],
      shape=(138240, 6))